Sequential Chats and Customer Onboarding

In [1]:
from getpass import getpass

def get_openai_api_key():
    return getpass("Enter your OpenAI API key: ")

OPENAI_API_KEY = get_openai_api_key()

llm_config = {
    "model": "gpt-6-luna",
    "api_key": OPENAI_API_KEY
}

print("OpenAI configuration created successfully!")

Enter your OpenAI API key:  ········


OpenAI configuration created successfully!


In [2]:
from autogen import ConversableAgent

onboarding_personal_information_agent = ConversableAgent(
    name="onboarding_personal_information_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's name and location.
    Do not ask for other information. Return 'TERMINATE' 
    when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

onboarding_topic_preference_agent = ConversableAgent(
    name="onboarding_topic_preference_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

customer_engagement_agent = ConversableAgent(
    name="customer_engagement_agent",  # Removed spaces
    system_message='''You are a helpful customer service agent.
    You are here to provide fun for the customer based on the user's
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)


C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Anaconda3\envs\agenticAI\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.18) which Google will stop supporting in new releases of google.cloud.resourcemanager_v3 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.resourcemanager_v3 past that date.
  warnings.warn(message, FutureWarning)


Creating tasks:
Now, you can craft a series of tasks to facilitate the onboarding process.

In [3]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history": False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

Start the onboarding process:
Note: You might get a slightly different response than what's shown in the video. Feel free to try different inputs, such as name, location, and preferences.

In [4]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)


********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product. Could you tell me your name and location?

--------------------------------------------------------------------------------


C:\Anaconda3\envs\agenticAI\Lib\site-packages\autogen\agentchat\chat.py:55: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Uma from Cuddalore


customer_proxy_agent (to onboarding_personal_information_agent):

Uma from Cuddalore

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics you are interested in reading about?
Context: 
{"name":"Uma","location":"Cuddalore"}

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  rule for chess


customer_proxy_agent (to onboarding_topic_preference_agent):

rule for chess

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Let's find something fun to read.
Context: 
{"name":"Uma","location":"Cuddalore"}
Uma, from Cuddalore, is interested in reading about chess rules.

--------------------------------------------------------------------------------
customer_engagement_agent (to customer_proxy_agent):

## Chess: A Tiny Battlefield of Big Ideas ♟️

Uma, here’s a quick chess read for a curious mind in Cuddalore: chess is a battle where the goal isn’t to capture every piece—it’s to trap the **king** so it has no safe move. That’s **checkmate**!

Each player takes one turn at a time:

- **Pawns** move forward, but ca

Print out the summary

In [5]:
summary=[]
for chat_result in chat_results:
    print(chat_result.summary)
    summary.append(chat_result.summary)
    print("\n")

{"name":"Uma","location":"Cuddalore"}


Uma, from Cuddalore, is interested in reading about chess rules.


Uma from Cuddalore is interested in chess rules. The response explained how the pieces move, the goal of checkmate, pawn promotion, castling, en passant, and stalemate.




In [6]:
summary[0]

'{"name":"Uma","location":"Cuddalore"}'

In [ ]:
#!pip install pandas

In [7]:
import pandas as pd
import ast

# Input string
#data_str = "{'name': 'Uma', 'location': 'Cuddalore'}"
data_str=summary[0]
# Convert string to dictionary
data_dict = ast.literal_eval(data_str)

# Convert dictionary to DataFrame
df = pd.DataFrame([data_dict])

df

,name,location
0,Uma,Cuddalore


In [8]:
df["TopicPreference"]=summary[1]
df

,name,location,TopicPreference
0,Uma,Cuddalore,"Uma, from Cuddalore, is interested in reading ..."


In [9]:
df["Enagement"]=summary[2]
df

,name,location,TopicPreference,Enagement
0,Uma,Cuddalore,"Uma, from Cuddalore, is interested in reading ...",Uma from Cuddalore is interested in chess rule...


In [ ]:
#!pip uninstall numpy pandas -y

In [ ]:
#!pip install numpy==1.26.4 pandas==2.2.3

In [ ]:
#!pip install gspread

In [ ]:
#!pip install oauth2client

In [10]:
def add_leads_to_google_sheet(df):
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials

    # Define the scope for Google Sheets API
    scope = ['https://www.googleapis.com/auth/spreadsheets']

    # Add credentials to the account
    creds = ServiceAccountCredentials.from_json_keyfile_name(
        'customer-onboarding-openai-97aa3c132d90.json', scope
    )

    # Authorize the client
    client = gspread.authorize(creds)

    # Specify the Google Sheet ID
    sheet_id = '1715fbc-io2EH8snyJRxbbCqE-6pKwonodUoFPHr-r38'

    # Open the spreadsheet using its ID
    spreadsheet = client.open_by_key(sheet_id)

    # Select the first worksheet (index 0)
    worksheet = spreadsheet.get_worksheet(0)

    # Append each row of the DataFrame to the worksheet
    for _, row in df.iterrows():
        # Convert row to a list and append to the sheet
        worksheet.append_row(row.tolist())

    print("Data added successfully!")
    return "added"


In [11]:
add_leads_to_google_sheet(df)

Data added successfully!


'added'

Print out the cost

In [12]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0, 'gpt-6-luna': {'cost': 0, 'prompt_tokens': 71, 'completion_tokens': 36, 'total_tokens': 107}}, 'usage_excluding_cached_inference': {'total_cost': 0}}


{'usage_including_cached_inference': {'total_cost': 0, 'gpt-6-luna': {'cost': 0, 'prompt_tokens': 74, 'completion_tokens': 71, 'total_tokens': 145}}, 'usage_excluding_cached_inference': {'total_cost': 0}}


{'usage_including_cached_inference': {'total_cost': 0, 'gpt-6-luna': {'cost': 0, 'prompt_tokens': 469, 'completion_tokens': 410, 'total_tokens': 879}}, 'usage_excluding_cached_inference': {'total_cost': 0}}


